# Data Cleaning & Preprocessing

**Objective:**
Based on the insights gathered during EDA, we need to clean the dataset. This involves handling missing Customer IDs, duplicates, returned/cancelled orders, and invalid prices to prepare the data for feature engineering.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [2]:
df = pd.read_excel('../data/raw/Online_Retail.xlsx')
print(f"Initial Shape: {df.shape}")

Initial Shape: (541909, 8)


## 2. Drop Missing Customer IDs
Customer segmentation is impossible without a customer identifier.

In [3]:
df_clean = df.dropna(subset=['CustomerID'])
print(f"Shape after dropping missing CustomerIDs: {df_clean.shape}")

Shape after dropping missing CustomerIDs: (406829, 8)


## 3. Remove Duplicates
Duplicate transactions can skew frequency and monetary metrics.

In [4]:
df_clean = df_clean.drop_duplicates()
print(f"Shape after dropping duplicates: {df_clean.shape}")

Shape after dropping duplicates: (401604, 8)


## 4. Handle Returns, Cancellations, and Negative Quantities
Invoices starting with 'C' indicate cancellations, and corresponding quantities are negative. For historical customer segmentation based on successful purchases, these should be removed.

In [5]:
df_clean = df_clean[df_clean['Quantity'] > 0]
print(f"Shape after keeping only positive quantities: {df_clean.shape}")

Shape after keeping only positive quantities: (392732, 8)


## 5. Handle Invalid Prices
Zero or negative prices represent bad data or non-purchase items (like manual adjustments or free gifts). We will keep only records with UnitPrice > 0.

In [6]:
df_clean = df_clean[df_clean['UnitPrice'] > 0]
print(f"Shape after dropping zero/negative prices: {df_clean.shape}")

Shape after dropping zero/negative prices: (392692, 8)


## 6. Create Revenue Column
Calculate the total revenue generated per line item.

In [7]:
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['UnitPrice']
print(f"Total Revenue across cleaned dataset: GBP {df_clean['Revenue'].sum():,.2f}")

Total Revenue across cleaned dataset: GBP 8,887,208.89


## 7. Save Cleaned Dataset
Export the clean dataset for the feature engineering and modeling phases.

In [8]:
df_clean.to_csv('../data/processed/cleaned_online_retail.csv', index=False)
print("Cleaned dataset saved successfully to 'data/processed/cleaned_online_retail.csv'.")

Cleaned dataset saved successfully to 'data/processed/cleaned_online_retail.csv'.


### Key Decisions Made
1. **Missing Customer IDs**: Dropped. Rationale: Cannot be mapped to a customer profile.
2. **Duplicates**: Dropped. Rationale: Prevents double-counting in monetary and frequency metrics.
3. **Negative/Zero Quantities**: Dropped. Rationale: We are segmenting based on successful buying behaviors, not return behaviors.
4. **Zero/Negative Unit Prices**: Dropped. Rationale: Non-revenue generating records skew monetary value.

*Note: Dropping returns removes the negative impact they have on revenue, which is a limitation of this approach. It may introduce slight bias by ignoring customers with high return rates, but it heavily simplifies and stabilizes the clustering problem.*